# Benchmark — JSL De-Identification (Masked & Obfuscated Outputs)

Uses our proven ONNX-based NER stack (`zeroshot_ner_deid_subentity_docwise_medium`) with
the JSL `LightDeIdentification` annotator to produce **masked** and **obfuscated** outputs —
the same output format as `clinical_deidentification_docwise_benchmark_medium`.

> **Why not the pretrained pipeline directly?**  
> `clinical_deidentification_docwise_benchmark_medium` uses a TensorFlow-based NER model
> that requires the `jnitensorflow` JNI native library. This library is not bundled in
> Spark NLP 6.3.x for Apple Silicon (arm64), causing a `UnsatisfiedLinkError` at load time.
> This notebook achieves the same result using ONNX-based components that run natively on arm64.

**Outputs:**
- `outputs/Benchmark_PlainText_Results.xlsx` — 20 test cases
- `outputs/Benchmark_XML_Results.xlsx` — 10 XML clinical documents


In [ ]:
import json, os, sys, re
from datetime import datetime, date as date_type

os.environ["JAVA_HOME"]             = "/opt/homebrew/opt/openjdk@11"
os.environ["PATH"]                  = "/opt/homebrew/opt/openjdk@11/bin:" + os.environ.get("PATH","")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

with open("spark_jsl.json") as f:
    license_keys = json.load(f)
locals().update(license_keys)
os.environ.update(license_keys)
print("License keys loaded. Python:", sys.executable)

In [ ]:
import sparknlp, sparknlp_jsl
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
import re, json, os, sys                     # restore after wildcard import
from datetime import datetime, date as date_type
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline, PipelineModel
import pandas as pd

from pyspark.sql import SparkSession
_s = SparkSession.getActiveSession()
if _s:
    _s.stop()
    import time; time.sleep(3)

spark = sparknlp_jsl.start(license_keys["SECRET"])
print("Spark NLP:", sparknlp.version(), "| JSL:", sparknlp_jsl.version())

In [ ]:
# ── NER stages (identical to Test_Cases_DeID.ipynb) ──────────────────────
documentAssembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

splitter = (InternalDocumentSplitter()
    .setInputCols("document").setOutputCol("sentence")
    .setSplitMode("recursive").setSplitPatterns([r"\s+|(?<=\G.{512})"])
    .setPatternsAreRegex(True).setChunkSize(512).setChunkOverlap(50)
    .setEnableSentenceIncrement(True))

tokenizer     = Tokenizer().setInputCols("sentence").setOutputCol("token")
tokenizer_doc = Tokenizer().setInputCols("document").setOutputCol("token_doc")

labels = ["DOCTOR","PATIENT","DATE_OF_BIRTH","DATE","CITY","STREET","STATE",
          "COUNTRY","PHONE","EMAIL","ZIP","USERNAME","ID","BIOID",
          "ORGANIZATION","MEDICAL_RECORD_NUMBER","SSN","AGE"]

zero_shot_ner = (PretrainedZeroShotNERChunker
    .pretrained("zeroshot_ner_deid_subentity_docwise_medium","en","clinical/models")
    .setInputCols("sentence").setOutputCol("ner_zero_shot")
    .setPredictionThreshold(0.7).setLabels(labels).setBatchSize(8))

zip_parser     = (ContextualParserModel.pretrained("zip_parser","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("zip_chunks"))
dob_parser     = (ContextualParserModel.pretrained("date_of_birth_parser","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("dob_chunks"))
email_matcher  = (RegexMatcherInternalModel.pretrained("email_matcher","en","clinical/models")
                  .setInputCols(["document"]).setOutputCol("email_chunks"))
country_matcher= (TextMatcherInternalModel.pretrained("country_matcher","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("country_chunks")
                  .setMergeOverlapping(True))

chunk_merge_ner = (ChunkMergeModel()
    .setInputCols("ner_zero_shot").setOutputCol("ner_merged")
    .setMergeOverlapping(True).setSelectionStrategy("DiverseLonger")
    .setResetSentenceIndices(True)
    .setReplaceDict({"DATE_OF_BIRTH":"DOB","MEDICAL_RECORD_NUMBER":"MEDICALRECORD"}))

chunk_merge_rules = (ChunkMergeModel()
    .setInputCols("zip_chunks","email_chunks","dob_chunks","country_chunks")
    .setOutputCol("rules_merged").setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

chunk_merge_final = (ChunkMergeModel()
    .setInputCols("ner_merged","rules_merged").setOutputCol("ner_chunk")
    .setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

# ── LightDeIdentification: masked output (<ENTITY_TYPE>) ─────────────────
deid_masked = (LightDeIdentification()
    .setInputCols(["ner_chunk","sentence"])
    .setOutputCol("masked")
    .setMode("mask"))

# ── LightDeIdentification: obfuscated output (fake realistic values) ──────
deid_obfuscated = (LightDeIdentification()
    .setInputCols(["ner_chunk","sentence"])
    .setOutputCol("obfuscated")
    .setMode("obfuscate")
    .setObfuscateDate(True)
    .setDateEntities(["DATE","DOB"])
    .setDays(30)
    .setAgeRanges([1,4,12,20,40,60,80]))

pipeline = Pipeline(stages=[
    documentAssembler, splitter, tokenizer, tokenizer_doc,
    zero_shot_ner, chunk_merge_ner,
    zip_parser, dob_parser, email_matcher, country_matcher,
    chunk_merge_rules, chunk_merge_final,
    deid_masked, deid_obfuscated,
])

model = pipeline.fit(spark.createDataFrame([[""]],("text",)))
print("Pipeline ready.")

---
## Part 1 — Plain Text: 20 Test Cases

In [ ]:
test_cases = [
    {"id":  1, "name": "Standard DOB label",
     "text": "Patient: John Smith. DOB: 03/22/1985. Seen by Dr. Alice Wong on 04/10/2024. ZIP: 94404."},
    {"id":  2, "name": "Born on (written phrase)",
     "text": "Emily Clarke was born on April 5, 1990. She visited our clinic on January 20, 2024. Address: 12 Oak St, Boston, MA 02115."},
    {"id":  3, "name": "Date of Birth (full label)",
     "text": "Name: Robert Thompson. Date of Birth: 15-07-1978. Appointment date: 2024-03-01. Postcode: LS8 4HE."},
    {"id":  4, "name": "D.O.B. abbreviation with dots",
     "text": "Patient Sarah Johnson. D.O.B.: 22.11.1965. Dr. Patel reviewed on 10.02.2024. SSN: 444-55-8888."},
    {"id":  5, "name": "Birth date (alternate phrasing)",
     "text": "Michael Brown, birth date 07/04/1972, was referred to Dr. Green on 15th March 2024. Contact: michael.brown@email.com."},
    {"id":  6, "name": "ISO date format (YYYY-MM-DD)",
     "text": "Jessica Taylor (DOB: 1988-09-14) attended on 2024-04-22. Treated by Dr. Karen Hill. MRN: 5247840."},
    {"id":  7, "name": "Born (short keyword)",
     "text": "William Davis, born 1969-12-30, called our helpline (07785 441229) after his visit on 2024-01-15."},
    {"id":  8, "name": "Birthday label",
     "text": "Linda Martinez, Birthday: 08/19/1980. Last seen: 03/05/2024. Username: lmartinez80. ZIP: 10001."},
    {"id":  9, "name": "UK postcode + written date",
     "text": "James Wilson was born on 3rd February 1975. His appointment was on 12th April 2024. Postcode: SW1A 1AA. Seen by Dr. Emma Clarke."},
    {"id": 10, "name": "Multiple patients in one record",
     "text": "Primary patient: Patricia Moore (DOB 05/30/1962). Emergency contact: Charles Anderson, born 12/01/1960. Both seen by Dr. Liu on 04/18/2024."},
    {"id": 11, "name": "Service date as 'admission date'",
     "text": "Barbara Thomas, DOB: 14.09.1970. Admission date: 22.01.2024. Treating physician: Dr. Samuel Okafor. Hospital: St Luke's Medical Center."},
    {"id": 12, "name": "US ZIP+4 format",
     "text": "Patient Daniel Foster, DOB: 04/11/1972. Address: 27 Oakfield Road, Manchester, M13 9PL. Zip: 94404-1234. Phone: (212) 555-7890."},
    {"id": 13, "name": "Doctor name must NOT be masked",
     "text": "John Smith was seen by Dr. Richard Feynman and Dr. Marie Curie on 03/10/2024. DOB: 1990-06-15. Email: j.smith@nhs.uk."},
    {"id": 14, "name": "SSN + MRN + Username",
     "text": "Patient Emily Clarke. SSN: 123-45-6789. MRN: MF-88201. Username: eclarke90. DOB: July 4, 1990. Visit: 04/01/2024."},
    {"id": 15, "name": "Multiple dates in one record",
     "text": "Robert Thompson, born 15-07-1978. Diagnosed on 01/10/2022. Follow-up scheduled for 05/20/2024. Reviewed by Dr. Aisha Patel. ZIP: 33101."},
    {"id": 16, "name": "International address + country",
     "text": "Sarah Johnson (DOB: 22/11/1965) is a resident of 14 Rue de Rivoli, Paris, France. She was seen remotely on 02/14/2024. Email: s.johnson@gmail.com."},
    {"id": 17, "name": "'Date seen' phrasing for service date",
     "text": "Michael Brown (born 07/04/1972). Date seen: April 15, 2024. Referred by Dr. Henry Chang. Phone: 0161 882 3300. Postcode: M1 3HF."},
    {"id": 18, "name": "'Seen on' + abbreviated month",
     "text": "Jessica Taylor, DOB 14 Sep 1988. Seen on 22 Apr 2024 by Dr. Olivia Scott. Address: 9888 Genesee Ave, USA. ZIP: 92037."},
    {"id": 19, "name": "Complex full clinical note",
     "text": ("Patient: William Davis, born 30/12/1969. NHS No: 882 441 9930. "
              "Referred to St. Mary's Hospital on 15 Jan 2024 by Dr. Thomas Nguyen. "
              "Address: 45 King Street, Leeds, LS1 2AB. Phone: 07785 123456. "
              "Email: w.davis@outlook.com. SSN: 321-76-5432. Username: wdavis69.")},
    {"id": 20, "name": "'Record date' vs 'DOB' disambiguation",
     "text": ("Record date: 04/28/2026. Patient: Linda Martinez. "
              "Date of Birth: 19/08/1980. Seen by Dr. Fatima Al-Rashid. "
              "MRN: 647390883. ZIP: 90210. Email: linda.m@hospital.org.")},
]

rows_data = [(tc["id"], tc["name"], tc["text"]) for tc in test_cases]
df = spark.createDataFrame(rows_data, ["test_id","test_name","text"])
result_df = model.transform(df)

@F.udf(StringType())
def entity_summary_udf(results, meta_list):
    if not results: return ""
    parts = []
    for i, chunk_text in enumerate(results):
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity","?") if meta else "?"
        parts.append(f"{chunk_text} → {entity}")
    return " | ".join(parts)

result_df = (result_df
    .withColumn("entities_detected",
        entity_summary_udf(F.col("ner_chunk.result"), F.col("ner_chunk.metadata")))
    .withColumn("masked_text",     F.concat_ws(" ", F.col("masked.result")))
    .withColumn("obfuscated_text", F.concat_ws(" ", F.col("obfuscated.result"))))

results_pd = (result_df
    .select("test_id","test_name","text","entities_detected","masked_text","obfuscated_text")
    .orderBy("test_id").toPandas())
results_pd.columns = ["#","Test Case","Original Text","Entities Detected","Masked Output","Obfuscated Output"]
print(f"Processed {len(results_pd)} test cases.")
results_pd

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

os.makedirs("outputs", exist_ok=True)
OUTPUT_PT = os.path.join("outputs","Benchmark_PlainText_Results.xlsx")

hdr_fill  = PatternFill("solid", fgColor="1F4E79")
hdr_font  = Font(bold=True, color="FFFFFF", size=11)
orig_fill = PatternFill("solid", fgColor="FFF2CC")
ent_fill  = PatternFill("solid", fgColor="E2EFDA")
msk_fill  = PatternFill("solid", fgColor="FCE4D6")
obf_fill  = PatternFill("solid", fgColor="DDEBF7")
alt_fill  = PatternFill("solid", fgColor="F2F2F2")

headers    = ["#","Test Case","Original Text","Entities Detected","Masked Output","Obfuscated Output"]
col_fills  = [None, None, orig_fill, ent_fill, msk_fill, obf_fill]
col_widths = [5, 30, 55, 60, 55, 55]

wb = openpyxl.Workbook()
ws = wb.active; ws.title = "Benchmark Results"

for ci,(h,w) in enumerate(zip(headers,col_widths),1):
    c = ws.cell(row=1,column=ci,value=h)
    c.font=hdr_font; c.fill=hdr_fill
    c.alignment=Alignment(wrap_text=True,vertical="center")
    ws.column_dimensions[get_column_letter(ci)].width=w
ws.row_dimensions[1].height=30

for ri, row in enumerate(results_pd.itertuples(index=False), 2):
    vals = list(row)
    for ci,(val,cfill) in enumerate(zip(vals,col_fills),1):
        cell = ws.cell(row=ri,column=ci,value=str(val) if val is not None else "")
        cell.alignment=Alignment(wrap_text=True,vertical="top")
        if cfill: cell.fill=cfill
        elif ri%2==0: cell.fill=alt_fill
    ws.row_dimensions[ri].height=60

ws.freeze_panes="A2"
wb.save(OUTPUT_PT)
print(f"Saved → {OUTPUT_PT}")

---
## Part 2 — XML Clinical Documents: 10 Files

In [ ]:
import xml.etree.ElementTree as ET, glob

XML_DIR   = "analysis"
xml_files = sorted(glob.glob(os.path.join(XML_DIR,"*.txt")))
print(f"Found {len(xml_files)} XML files")

def extract_xml_texts(path):
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as e:
        return [], str(e)
    return [(el.tag.split("}")[-1] if "}" in el.tag else el.tag, el.text.strip())
            for el in root.iter() if el.text and el.text.strip()], None

xml_sheet_data = {}

for fpath in xml_files:
    fname    = os.path.basename(fpath)
    segs, err = extract_xml_texts(fpath)
    if err:
        xml_sheet_data[fname] = [{"tag":"ERROR","original":err,"entities":"","masked":"","obfuscated":""}]
        continue

    unique_texts = list({txt for _,txt in segs})
    print(f"  {fname}: {len(segs)} nodes → {len(unique_texts)} unique")

    xml_df     = spark.createDataFrame([(i,t) for i,t in enumerate(unique_texts)],["idx","text"])
    xml_result = model.transform(xml_df)
    xml_result = (xml_result
        .withColumn("entities_detected",
            entity_summary_udf(F.col("ner_chunk.result"),F.col("ner_chunk.metadata")))
        .withColumn("masked_text",     F.concat_ws(" ",F.col("masked.result")))
        .withColumn("obfuscated_text", F.concat_ws(" ",F.col("obfuscated.result"))))

    lookup = {r["text"]: {"entities":r["entities_detected"]or"",
                           "masked":r["masked_text"]or"",
                           "obfuscated":r["obfuscated_text"]or""}
              for r in xml_result.select("text","entities_detected","masked_text","obfuscated_text").collect()}

    file_rows = []
    for tag,txt in segs:
        d = lookup.get(txt,{})
        if d.get("entities") or (d.get("masked") and d.get("masked")!=txt):
            file_rows.append({"tag":tag,"original":txt,
                               "entities":d.get("entities",""),
                               "masked":d.get("masked",""),
                               "obfuscated":d.get("obfuscated","")})

    xml_sheet_data[fname] = file_rows
    print(f"    → {len(file_rows)} PHI nodes")

print("\nXML processing complete.")

In [ ]:
print(f"{'File':<15}  {'PHI nodes':>10}")
print("-"*28)
for fname,rows in xml_sheet_data.items():
    print(f"  {fname:<13}  {len(rows):>10}")
print("-"*28)
print(f"  {'TOTAL':<13}  {sum(len(r) for r in xml_sheet_data.values()):>10}")

In [ ]:
OUTPUT_XML = os.path.join("outputs","Benchmark_XML_Results.xlsx")

headers_x   = ["XML Tag","Original Text","Entities Detected","Masked Output","Obfuscated Output"]
col_widths_x = [20,55,60,55,55]
col_fills_x  = [None,orig_fill,ent_fill,msk_fill,obf_fill]

wb2 = openpyxl.Workbook()
del wb2[wb2.sheetnames[0]]
summary_rows = []

for fname,rows in xml_sheet_data.items():
    sname = fname.replace(".txt","").replace(".xml","")[:31]
    ws2 = wb2.create_sheet(title=sname)
    for ci,(h,w) in enumerate(zip(headers_x,col_widths_x),1):
        c=ws2.cell(row=1,column=ci,value=h)
        c.font=hdr_font; c.fill=hdr_fill
        c.alignment=Alignment(wrap_text=True,vertical="center")
        ws2.column_dimensions[get_column_letter(ci)].width=w
    ws2.row_dimensions[1].height=30
    for ri,row in enumerate(rows,2):
        vals=[row["tag"],row["original"],row["entities"],row["masked"],row["obfuscated"]]
        for ci,(val,cfill) in enumerate(zip(vals,col_fills_x),1):
            cell=ws2.cell(row=ri,column=ci,value=str(val) if val else "")
            cell.alignment=Alignment(wrap_text=True,vertical="top")
            if cfill: cell.fill=cfill
            elif ri%2==0: cell.fill=alt_fill
        ws2.row_dimensions[ri].height=55
    ws2.freeze_panes="A2"
    summary_rows.append({"file":fname,"phi":len(rows)})

ws_s=wb2.create_sheet("Summary",0)
for ci,h in enumerate(["File","PHI Nodes Detected"],1):
    c=ws_s.cell(row=1,column=ci,value=h)
    c.font=hdr_font; c.fill=hdr_fill
    ws_s.column_dimensions[get_column_letter(ci)].width=25
ws_s.row_dimensions[1].height=28
for ri,d in enumerate(summary_rows,2):
    ws_s.cell(row=ri,column=1,value=d["file"])
    ws_s.cell(row=ri,column=2,value=d["phi"])
tr=len(summary_rows)+2
ws_s.cell(row=tr,column=1,value="TOTAL").font=Font(bold=True)
ws_s.cell(row=tr,column=2,value=sum(d["phi"] for d in summary_rows)).font=Font(bold=True)

wb2.save(OUTPUT_XML)
print(f"Saved → {OUTPUT_XML}")

In [ ]:
spark.stop()
print("Spark stopped — license released.")